<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day9_Logistics_Invoice_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install azure-storage-blob azure-ai-documentintelligence pymupdf pypdf azure-cosmos


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.5/129.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.7/455.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 10.5 MB/s eta 0:00:00


In [10]:
import os

# The secret must be configured in Colab Secrets or your approved environment.
# Do not print the URL because it contains a bearer-like token.
from google.colab import userdata


blob_sas_url = userdata.get("BLOB_SAS_URL")


if not blob_sas_url:
    raise ValueError("BLOB_SAS_URL is missing.")


print("Blob URL configured successfully.")


Blob URL configured successfully.


In [11]:
from azure.storage.blob import BlobClient
import os

blob_sas_url = blob_sas_url.strip().strip('"').strip("'")   # remove stray spaces/quotes
blob_client = BlobClient.from_blob_url(blob_sas_url)

local_pdf_path = "/content/invoice.pdf"
data = blob_client.download_blob().readall()                 # download first
with open(local_pdf_path, "wb") as local_file:               # write only if it succeeded
    local_file.write(data)

print("Download complete:", local_pdf_path)
print("File size (bytes):", os.path.getsize(local_pdf_path))

Download complete: /content/invoice.pdf
File size (bytes): 27738


In [12]:
import fitz

with fitz.open(local_pdf_path) as pdf:
    page_texts = []
    for page_number, page in enumerate(pdf, start=1):
        text = page.get_text()
        page_texts.append(text)
        print(f"--- Page {page_number} ---")
        print(text[:2000])


--- Page 1 ---
SUPPLIER INVOICE
FICTIONAL TRAINING DOCUMENT — NOT A REAL INVOICE
Supplier
Northstar Freight Supplies
Invoice number
INV-LOG-2048
Invoice date
18 September 2026
Purchase order
PO-7781
Invoice Summary
Description
Amount (INR)
Freight and logistics services
12,500.00
Tax
2,250.00
Total
14,750.00
Amount in words: Fourteen thousand seven hundred fifty Indian rupees only.
This document was created solely for a training exercise on PDF text extraction, OCR, Azure AI Document
Intelligence, JSON processing, and Cosmos DB. It does not represent a real supplier, transaction, tax invoice, or
claim for payment.



In [13]:
from pypdf import PdfReader

reader = PdfReader(local_pdf_path)
pypdf_text = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pypdf_text.append(text)
    print(f"--- Page {page_number} ---")
    print(text[:2000])


--- Page 1 ---
SUPPLIER INVOICE
FICTIONAL TRAINING DOCUMENT — NOT A REAL INVOICE
Supplier
Northstar Freight Supplies
Invoice number
INV-LOG-2048
Invoice date
18 September 2026
Purchase order
PO-7781
Invoice Summary
Description
Amount (INR)
Freight and logistics services
12,500.00
Tax
2,250.00
Total
14,750.00
Amount in words: Fourteen thousand seven hundred fifty Indian rupees only.
This document was created solely for a training exercise on PDF text extraction, OCR, Azure AI Document
Intelligence, JSON processing, and Cosmos DB. It does not represent a real supplier, transaction, tax invoice, or
claim for payment.



In [16]:
from google.colab import userdata


endpoint = userdata.get("DOCUMENTINTELLIGENCE_ENDPOINT")
document_intelligence_key = userdata.get("DOCUMENTINTELLIGENCE_API_KEY")


if not endpoint or not document_intelligence_key:
    raise ValueError("Configure the Document Intelligence endpoint and key securely.")


print("Document Intelligence configuration loaded.")


Document Intelligence configuration loaded.


In [17]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential

di_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(document_intelligence_key)
)

with open(local_pdf_path, "rb") as pdf_file:
    poller = di_client.begin_analyze_document(
        "prebuilt-read",
        body=pdf_file,
        content_type="application/pdf"
    )

analysis_result = poller.result()

extracted_text = analysis_result.content or ""
print(extracted_text[:5000])


SUPPLIER INVOICE
FICTIONAL TRAINING DOCUMENT - NOT A REAL INVOICE
Supplier
Northstar Freight Supplies
Invoice number
INV-LOG-2048
Invoice date
18 September 2026
Purchase order
PO-7781
Invoice Summary
Description
Amount (INR)
Freight and logistics services
12,500.00
Tax
2,250.00
Total
14,750.00
Amount in words: Fourteen thousand seven hundred fifty Indian rupees only.
This document was created solely for a training exercise on PDF text extraction, OCR, Azure AI Document Intelligence, JSON processing, and Cosmos DB. It does not represent a real supplier, transaction, tax invoice, or claim for payment.


In [18]:
import json
from pathlib import Path

record = {
    "id": "INV-LOG-2048",
    "supplier": "Northstar Freight Supplies",
    "document_type": "supplier_invoice",
    "source_blob": "invoice.pdf",
    "extracted_text": extracted_text,
    "review_status": "pending"
}

json_path = "/content/invoice_extraction.json"

with open(json_path, "w", encoding="utf-8") as json_file:
    json.dump(record, json_file, ensure_ascii=False, indent=2)

print("JSON saved:", json_path)
print(json.dumps(record, ensure_ascii=False, indent=2)[:1500])


JSON saved: /content/invoice_extraction.json
{
  "id": "INV-LOG-2048",
  "supplier": "Northstar Freight Supplies",
  "document_type": "supplier_invoice",
  "source_blob": "invoice.pdf",
  "extracted_text": "SUPPLIER INVOICE\nFICTIONAL TRAINING DOCUMENT - NOT A REAL INVOICE\nSupplier\nNorthstar Freight Supplies\nInvoice number\nINV-LOG-2048\nInvoice date\n18 September 2026\nPurchase order\nPO-7781\nInvoice Summary\nDescription\nAmount (INR)\nFreight and logistics services\n12,500.00\nTax\n2,250.00\nTotal\n14,750.00\nAmount in words: Fourteen thousand seven hundred fifty Indian rupees only.\nThis document was created solely for a training exercise on PDF text extraction, OCR, Azure AI Document Intelligence, JSON processing, and Cosmos DB. It does not represent a real supplier, transaction, tax invoice, or claim for payment.",
  "review_status": "pending"
}


In [19]:
with open(local_pdf_path, "rb") as pdf_file:
    invoice_poller = di_client.begin_analyze_document(
        "prebuilt-invoice",
        body=pdf_file,
        content_type="application/pdf"
    )

invoice_result = invoice_poller.result()

for document in (invoice_result.documents or []):
    fields = document.fields or {}
    for field_name, field in fields.items():
        value = getattr(field, "content", None)
        confidence = getattr(field, "confidence", None)
        print(field_name, "=", value, "| confidence =", confidence)


InvoiceDate = 18 September 2026 | confidence = 0.93
InvoiceId = INV-LOG-2048 | confidence = 0.985
InvoiceTotal = 14,750.00 | confidence = 0.984
Items = None | confidence = None
PurchaseOrder = PO-7781 | confidence = 0.984
TotalTax = 2,250.00 | confidence = 0.985
VendorName = Northstar Freight Supplies | confidence = 0.888


In [22]:
from google.colab import userdata
from azure.cosmos import CosmosClient

cosmos_connection_string = userdata.get("COSMOS_CONNECTION_STRING")
if not cosmos_connection_string:
    raise ValueError("COSMOS_CONNECTION_STRING is missing.")

cosmos_connection_string = cosmos_connection_string.strip().strip('"').strip("'")
print("Starts with AccountEndpoint=:", cosmos_connection_string.startswith("AccountEndpoint="))

cosmos_client = CosmosClient.from_connection_string(cosmos_connection_string)
database = cosmos_client.get_database_client("logistics-lab")
container = database.get_container_client("invoice-records")

cosmos_record = {
    "id": "INV-LOG-2049",
    "supplier": "Northstar Freight Supplies",
    "documentType": "supplier_invoice",
    "invoiceDate": "2026-09-18",
    "currency": "INR",
    "total": 14750.00,
    "reviewStatus": "pending"
}

saved = container.upsert_item(cosmos_record)
print("Saved record ID:", saved["id"])

Starts with AccountEndpoint=: True


CosmosResourceNotFoundError: (NotFound) Message: {"Errors":["Resource Not Found. Learn more: https://aka.ms/cosmosdb-tsg-not-found"]}
ActivityId: fac9bb4a-5eb0-4c67-9f83-9968d16a92c4, Request URI: /apps/9d8fd7c5-1c06-40d7-a443-dd8c607c7ad3/services/0606fb38-15b8-4f00-a9bd-c23600d42027/partitions/fd161ce4-10f4-44f2-bd2e-cd2ac17d6829/replicas/134364365853987822s, RequestStats: , SDK: Microsoft.Azure.Documents.Common/2.14.0
Code: NotFound
Message: Message: {"Errors":["Resource Not Found. Learn more: https://aka.ms/cosmosdb-tsg-not-found"]}
ActivityId: fac9bb4a-5eb0-4c67-9f83-9968d16a92c4, Request URI: /apps/9d8fd7c5-1c06-40d7-a443-dd8c607c7ad3/services/0606fb38-15b8-4f00-a9bd-c23600d42027/partitions/fd161ce4-10f4-44f2-bd2e-cd2ac17d6829/replicas/134364365853987822s, RequestStats: , SDK: Microsoft.Azure.Documents.Common/2.14.0

In [23]:
for db in cosmos_client.list_databases():
    print("DATABASE :", db["id"])
    for c in cosmos_client.get_database_client(db["id"]).list_containers():
        print("   CONTAINER:", c["id"], "| partition key:", c["partitionKey"]["paths"])

DATABASE : logistics-lab
   CONTAINER: invoice-record | partition key: ['/supplier']


In [24]:
database = cosmos_client.get_database_client("logistics-lab")
container = database.get_container_client("invoice-record")   # singular, as it exists in Azure

In [25]:
cosmos_record = {
    "id": "INV-LOG-2049",
    "supplier": "Northstar Freight Supplies",
    "documentType": "supplier_invoice",
    "invoiceDate": "2026-09-18",
    "currency": "INR",
    "total": 14750.00,
    "reviewStatus": "pending"
}

saved = container.upsert_item(cosmos_record)
print("Saved record ID:", saved["id"])

Saved record ID: INV-LOG-2049


In [26]:
for item in container.query_items(
    query="SELECT c.id, c.supplier, c.total, c.reviewStatus FROM c WHERE c.reviewStatus = 'pending'",
    enable_cross_partition_query=True
):
    print(item)

{'id': 'INV-LOG-2048', 'supplier': 'Northstar Freight Supplies', 'total': 14750, 'reviewStatus': 'pending'}
{'id': 'INV-LOG-2049', 'supplier': 'Northstar Freight Supplies', 'total': 14750, 'reviewStatus': 'pending'}
